# Belief-Agent v2: Negotiation Scenarios

Resolve conflicting goals between agents through structured negotiation.

In [ ]:
import json, sys
sys.path.insert(0, "..")
from belief_agent import BeliefAgent, BeliefState
from belief_agent.negotiation import Goal, negotiate, find_tradeoffs, suggest_compromise

## Scenario: Architecture Decision

**Engineer** wants microservices. **Product Manager** wants speed.

In [ ]:
def engineer_llm(messages):
    p = messages[-1]["content"]
    if "Rank the following goals" in p:
        return json.dumps([{"index": 0, "rank": 2}, {"index": 1, "rank": 1}, {"index": 2, "rank": 3}])
    return json.dumps({"goal_index": 0, "action": "Use microservices", "satisfaction": 0.7, "tradeoffs": ["complexity"]})

def pm_llm(messages):
    p = messages[-1]["content"]
    if "Rank the following goals" in p:
        return json.dumps([{"index": 0, "rank": 1}, {"index": 1, "rank": 2}, {"index": 2, "rank": 3}])
    return json.dumps({"goal_index": 1, "action": "Ship monolith first", "satisfaction": 0.9, "tradeoffs": ["tech debt"]})

engineer = BeliefAgent(llm_call=engineer_llm, name="Engineer")
pm = BeliefAgent(llm_call=pm_llm, name="ProductManager")

goals = [
    Goal("Scalability", priority=0.8),
    Goal("Speed to market", priority=0.9),
    Goal("Low cost", priority=0.5),
]

result = negotiate([engineer, pm], "Architecture", goals, engineer_llm)
print(f"Issue: {result.issue}")
print(f"Consensus action: {result.consensus_action}")
print(f"Satisfaction: {result.consensus_satisfaction:.2f}")
print(f"Unresolved: {result.unresolved_goals}")

## Find tradeoffs between conflicting goals

In [ ]:
def tradeoff_llm(messages):
    return json.dumps({"tradeoffs": ["Use serverless for auto-scaling", "Start with monolith, extract later"]})

ga = Goal("Scalability", priority=0.9)
gb = Goal("Time to market", priority=0.9)
tradeoffs = find_tradeoffs(engineer, ga, gb, tradeoff_llm)
print("Tradeoffs:")
for t in tradeoffs:
    print(f"  - {t}")

## Suggest a compromise

In [ ]:
from belief_agent.negotiation import NegotiationProposal

def comp_llm(messages):
    return json.dumps({"compromise": "Build modular monolith with clear bounded contexts"})

proposals = [
    NegotiationProposal(Goal("Scalability"), "Microservices", 0.7, ["complexity"]),
    NegotiationProposal(Goal("Speed"), "Monolith first", 0.9, ["tech debt"]),
]
compromise = suggest_compromise(proposals, comp_llm)
print(f"Suggested compromise: {compromise}")